# Assignment #4 (demo). Exploring OLS, Lasso and Random Forest in a regression task

In [2]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Lasso, LassoCV, LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GridSearchCV, cross_val_score, train_test_split
from sklearn.preprocessing import StandardScaler 

### We are working with UCI wine quality dataset

In [3]:
DATA_PATH = "https://raw.githubusercontent.com/Yorko/mlcourse.ai/main/data/"

In [4]:
data = pd.read_csv(DATA_PATH + "winequality-white.csv", sep=";")
data.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.0,0.27,0.36,20.7,0.045,45.0,170.0,1.0010,3.00,0.45,8.8,6
1,6.3,0.30,0.34,1.6,0.049,14.0,132.0,0.9940,3.30,0.49,9.5,6
2,8.1,0.28,0.40,6.9,0.050,30.0,97.0,0.9951,3.26,0.44,10.1,6
3,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,9.9,6
4,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,9.9,6


In [5]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4898 entries, 0 to 4897
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         4898 non-null   float64
 1   volatile acidity      4898 non-null   float64
 2   citric acid           4898 non-null   float64
 3   residual sugar        4898 non-null   float64
 4   chlorides             4898 non-null   float64
 5   free sulfur dioxide   4898 non-null   float64
 6   total sulfur dioxide  4898 non-null   float64
 7   density               4898 non-null   float64
 8   pH                    4898 non-null   float64
 9   sulphates             4898 non-null   float64
 10  alcohol               4898 non-null   float64
 11  quality               4898 non-null   int64  
dtypes: float64(11), int64(1)
memory usage: 459.3 KB


### Separate the target feature, split data in 7:3 proportion (30% form a holdout set, use random_state=17), and preprocess data with StandardScaler.

In [6]:
y = data["quality"]
X = data.drop("quality", axis=1)

X_train, X_holdout, y_train, y_holdout = train_test_split(
    X, y , test_size=0.3, random_state=17
)

In [7]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_holdout_scaled = scaler.fit_transform(X_holdout)

## Linear Regression

### Train a simple linear regression model (Ordinary Least Squares).

In [8]:
linreg = LinearRegression()
linreg.fit(X_train_scaled, y_train) # assign the weights

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [9]:
linreg_predict = linreg.predict(X_holdout_scaled) # returns the observations - y

In [10]:
# Question 1: What are mean squared errors of model predictions on train and holdout sets?
# Mean Squared Error (MSE) is a common metric used to measure how well a model's predictions match the actual values.
# OLS is the method used to find the best-fitting line (or hyperplane) by minimizing the sum of squared errors (SSE).
# MSE = SSE/n where n is the number of samples(data points)

mean_squared_error_train = mean_squared_error(y_train, linreg.predict(X_train_scaled))
mean_squared_error_holdout = mean_squared_error(y_holdout, linreg.predict(X_holdout_scaled))

mean_squared_error_train, mean_squared_error_holdout


(0.5580606489803572, 0.5820851719476797)

In [11]:
# Question 2: Which feature does this linear regression model treat as the most influential on wine quality?

# convert to dataframe and sort by absolute value of weights(coef_)

df = pd.DataFrame({
    "coeff" : linreg.coef_,
    "abs_coeff" : np.abs(linreg.coef_)
},
index=data.columns.drop("quality"))
df.sort_values("abs_coeff", ascending=False)


,coeff,abs_coeff
density,-0.665720,0.665720
residual sugar,0.538164,0.538164
volatile acidity,-0.192260,0.192260
pH,0.150036,0.150036
alcohol,0.129533,0.129533
fixed acidity,0.097822,0.097822
sulphates,0.062053,0.062053
free sulfur dioxide,0.042180,0.042180
total sulfur dioxide,0.014304,0.014304
chlorides,0.008127,0.008127


# Lasso Regression

**Lasso Regression (Least Absolute Shrinkage and Selection Operator)** is a type of **linear regression** that uses **L1 regularization** to improve model performance and automatically perform **feature selection**.

It is used when the target variable is **continuous** (e.g., predicting house prices, sales, or temperatures), especially when the dataset contains many input features.

## How it works

Like linear regression, Lasso predicts the target using a linear equation:

$$
\hat{y} = \beta_0 + \beta_1x_1 + \beta_2x_2 + \cdots + \beta_px_p
$$

However, instead of minimizing only the prediction error, Lasso adds an **L1 penalty** to the loss function:

$$
\text{Loss} = \sum_{i=1}^{n}(y_i-\hat{y}*i)^2 + \lambda \sum*{j=1}^{p}|\beta_j|
$$

Where:

* (y_i) = Actual value
* (\hat{y}_i) = Predicted value
* (\beta_j) = Model coefficients
* (\lambda) = Regularization parameter

The first term minimizes prediction error, while the second term penalizes large coefficient values.

## What is L1 Regularization?

L1 regularization adds the **absolute values of the coefficients** to the loss function.

As the value of **(\lambda)** increases:

* Coefficients become smaller.
* Some coefficients become **exactly zero**.
* Features with zero coefficients are effectively removed from the model.

This makes Lasso useful for **automatic feature selection**.

## Advantages

* Performs automatic feature selection.
* Reduces overfitting.
* Produces simpler and more interpretable models.
* Works well when many features are irrelevant.

## Disadvantages

* May remove useful features if regularization is too strong.
* Can be unstable when highly correlated features exist (it may choose one and discard others).
* Usually requires feature scaling before training.

## Choosing the Regularization Parameter ((\lambda))

* **Small (\lambda):**

  * Behaves similarly to ordinary linear regression.
  * Little regularization.
  * Higher risk of overfitting.

* **Large (\lambda):**

  * Strong regularization.
  * More coefficients become zero.
  * Higher risk of underfitting if too large.

The best value is typically selected using **cross-validation**.

## Applications

Lasso regression is commonly used in:

* Predicting house prices
* Sales forecasting
* Medical data analysis
* Financial modeling
* Gene selection in bioinformatics
* Any regression problem with many features

## Comparison with Related Algorithms

| Algorithm                | Task           | Regularization     |
| ------------------------ | -------------- | ------------------ |
| Linear Regression        | Regression     | None               |
| Lasso Regression         | Regression     | L1                 |
| Ridge Regression         | Regression     | L2                 |
| Logistic Regression      | Classification | None (or optional) |
| Logistic Regression + L1 | Classification | L1 (Lasso)         |
| Logistic Regression + L2 | Classification | L2 (Ridge)         |

## Lasso vs Ridge Regression

| Feature                      | Lasso Regression | Ridge Regression |
| ---------------------------- | ---------------- | ---------------- |
| Regularization               | L1               | L2               |
| Feature Selection            | Yes              | No               |
| Coefficients Can Become Zero | Yes              | No               |
| Handles Multicollinearity    | Good             | Excellent        |
| Model Complexity             | Simpler          | More Complex     |

## Example

Suppose you want to predict a house's price using:

* Size
* Number of bedrooms
* Age
* Distance to city
* Garage size
* Swimming pool
* Garden area

A Lasso model may determine that **garage size** and **garden area** contribute very little to predicting the price. It sets their coefficients to **0**, effectively removing them from the model while using the remaining important features to make predictions.

## Key Takeaways

* Lasso Regression is a **regression algorithm**, not a classification algorithm.
* It extends linear regression by adding **L1 regularization**.
* The L1 penalty shrinks coefficients and can reduce some to **exactly zero**, performing **automatic feature selection**.
* It helps reduce overfitting and creates simpler, more interpretable models.
* For **classification**, the equivalent approach is **Logistic Regression with L1 regularization**, often referred to as **L1-regularized logistic regression**.


### Train a LASSO model with alpha = 0.01 (weak regularization) and scaled data. Again, set random_state=17.

In [12]:
lasso1 = Lasso(alpha=0.01, random_state=17)
lasso1.fit(X_train_scaled, y_train)

,"alpha alpha: float, default=1.0Constant that multiplies the L1 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Lasso` object is not advised.Instead, you should use the :class:`LinearRegression` object.",0.01
,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"precompute precompute: bool or array-like of shape (n_features, n_features), default=FalseWhether to use a precomputed Gram matrix to speed upcalculations. The Gram matrix can also be passed as argument.For sparse input this option is always ``False`` to preserve sparsity.",False
,"copy_X copy_X: bool, default=TrueIf ``True``, X will be copied; else, it may be overwritten.",True
,"max_iter max_iter: int, default=1000The maximum number of iterations.",1000
,"tol tol: float, default=1e-4The tolerance for the optimization: if the updates are smaller or equal to``tol``, the optimization code checks the dual gap for optimality and continuesuntil it is smaller or equal to ``tol``, see Notes below.",0.0001
,"warm_start warm_start: bool, default=FalseWhen set to ``True``, reuse the solution of the previous call to fit asinitialization, otherwise, just erase the previous solution.See :term:`the Glossary `.",False
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive.",False
,"random_state random_state: int, RandomState instance, default=NoneThe seed of the pseudo random number generator that selects a randomfeature to update. Used when ``selection`` == 'random'.Pass an int for reproducible output across multiple function calls.See :term:`Glossary `.",17
,"selection selection: {'cyclic', 'random'}, default='cyclic'If set to 'random', a random coefficient is updated every iterationrather than looping over features sequentially by default. This(setting to 'random') often leads to significantly faster convergenceespecially when tol is higher than 1e-4.",'cyclic'


In [13]:
  # Which feature is the least informative in predicting wine quality, according to this LASSO model?

lasso_coeff1 = pd.DataFrame({
    "coeff" : lasso1.coef_,
    "coeff_abs" : np.abs(lasso1.coef_)
},
index=data.columns.drop("quality")) # insted of index being 0,1,2 it will be replaced with column names

lasso_coeff1.sort_values("coeff_abs", ascending=True)

,coeff,coeff_abs
fixed acidity,-0.000000,0.000000
citric acid,-0.000000,0.000000
total sulfur dioxide,-0.000000,0.000000
chlorides,-0.002747,0.002747
sulphates,0.029722,0.029722
free sulfur dioxide,0.043088,0.043088
pH,0.067277,0.067277
volatile acidity,-0.188479,0.188479
density,-0.235492,0.235492
residual sugar,0.256363,0.256363


In [14]:
# Train LassoCV with random_state=17 to choose the best value of in 5-fold cross-validation.

alphas = np.logspace(-6, 2, 200)

lasso_cv = LassoCV(alphas=alphas, random_state=17, cv=5 )
lasso_cv.fit(X_train_scaled, y_train)

,"eps eps: float, default=1e-3Length of the path. ``eps=1e-3`` means that``alpha_min / alpha_max = 1e-3``.",0.001
,"n_alphas n_alphas: int, default=100Number of alphas along the regularization path... deprecated:: 1.7 `n_alphas` was deprecated in 1.7 and will be removed in 1.9. Use `alphas` instead.",'deprecated'
,"alphas alphas: array-like or int, default=NoneValues of alphas to test along the regularization path.If int, `alphas` values are generated automatically.If array-like, list of alpha values to use... versionchanged:: 1.7 `alphas` accepts an integer value which removes the need to pass `n_alphas`... deprecated:: 1.7 `alphas=None` was deprecated in 1.7 and will be removed in 1.9, at which point the default value will be set to 100.",array([1.0000...00000000e+02])
,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto false, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"precompute precompute: 'auto', bool or array-like of shape (n_features, n_features), default='auto'Whether to use a precomputed Gram matrix to speed upcalculations. If set to ``'auto'`` let us decide. The Grammatrix can also be passed as argument.",'auto'
,"max_iter max_iter: int, default=1000The maximum number of iterations.",1000
,"tol tol: float, default=1e-4The tolerance for the optimization: if the updates are smaller or equal to``tol``, the optimization code checks the dual gap for optimality and continuesuntil it is smaller or equal to ``tol``.",0.0001
,"copy_X copy_X: bool, default=TrueIf ``True``, X will be copied; else, it may be overwritten.",True
,"cv cv: int, cross-validation generator or iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross-validation,- int, to specify the number of folds.- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For int/None inputs, :class:`~sklearn.model_selection.KFold` is used.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: bool or int, default=FalseAmount of verbosity.",False
,"n_jobs n_jobs: int, default=NoneNumber of CPUs to use during the cross validation.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None


In [15]:
lasso_cv.alpha_ # The amount of penalization chosen by cross validation.
lasso_cv.coef_ # weights assigned according to alpha_

array([ 0.09329524, -0.19204856, -0.        ,  0.52688307,  0.00693292,
        0.04269847,  0.01296898, -0.64816097,  0.14654889,  0.0609392 ,
        0.13711478])

In [16]:
# Question 3: Which feature is the least informative in predicting wine quality, according to the tuned LASSO model?

lasso_cv_coeff = pd.DataFrame({
    "coeff" : lasso_cv.coef_,
    "coeff_abs" : np.abs(lasso_cv.coef_)
},
index=data.columns.drop("quality"))

lasso_cv_coeff.sort_values("coeff_abs", ascending=True)

,coeff,coeff_abs
citric acid,-0.000000,0.000000
chlorides,0.006933,0.006933
total sulfur dioxide,0.012969,0.012969
free sulfur dioxide,0.042698,0.042698
sulphates,0.060939,0.060939
fixed acidity,0.093295,0.093295
alcohol,0.137115,0.137115
pH,0.146549,0.146549
volatile acidity,-0.192049,0.192049
residual sugar,0.526883,0.526883


In [17]:
# Question 4: What are mean squared errors of tuned LASSO predictions on train and holdout sets?

mean_squared_error_lasso_train = mean_squared_error(y_train, lasso_cv.predict(X_train_scaled))
mean_squared_error_lasso_holdout = mean_squared_error(y_holdout, lasso_cv.predict(X_holdout_scaled))

mean_squared_error_lasso_train, mean_squared_error_holdout

(0.558070014187378, 0.5820851719476797)

## Is the scores good?

These scores **look reasonable**, but whether they are "good" depends on what you're comparing them against.

Your results are:

* **Training MSE:** 0.5581
* **Holdout MSE:** 0.5821

### 1. The train and holdout scores are very close ✅

This is a good sign.

```text
Training MSE = 0.558
Holdout MSE = 0.582
Difference   = 0.024
```

Since the holdout error is only slightly higher than the training error, your model is **generalizing well**. It doesn't appear to be overfitting.

---

### 2. Is an MSE of 0.58 good?

By itself, it's hard to tell.

MSE has units of the **target squared**, so you should compare it with:

* a baseline model (e.g., always predict the mean),
* another regression model (Linear Regression, Ridge, etc.),
* or convert it to RMSE, which is easier to interpret.

For example:

```python
from sklearn.metrics import mean_squared_error
import numpy as np

rmse_train = np.sqrt(mean_squared_error_lasso_train)
rmse_holdout = np.sqrt(mean_squared_error_lasso_holdout)

print(rmse_train, rmse_holdout)
```

This gives approximately

```text
Train RMSE   ≈ 0.747
Holdout RMSE ≈ 0.763
```

If your wine quality scores are integers like **3–8**, then an average prediction error of about **0.76 quality points** is generally quite reasonable.

---

### 3. Compare with other models

The real question is whether LASSO improved things.

For example:

| Model                   | Holdout MSE |
| ----------------------- | ----------: |
| Baseline (predict mean) |        0.85 |
| Linear Regression       |        0.57 |
| Ridge                   |        0.56 |
| **LASSO**               |    **0.58** |

If your LASSO achieves about the same performance while using fewer features (because it sets some coefficients to zero), that's often considered a good trade-off.

### One small correction

Your code defines:

```python
mean_squared_error_lasso_holdout = ...
```

but then prints:

```python
mean_squared_error_holdout
```

Those variable names don't match. If your notebook still produced output, it probably means `mean_squared_error_holdout` was defined earlier. To avoid confusion, print the variable you just computed:

```python
mean_squared_error_lasso_train, mean_squared_error_lasso_holdout
```

If you also have the **Linear Regression** or **Ridge** MSE, I can help you compare them and determine which model is best.


# Random Forest

### Train a Random Forest with out-of-the-box parameters, setting only random_state to be 17.

In [18]:
forest = RandomForestRegressor(random_state=17)
forest.fit(X_train_scaled, y_train) # we use scaled data to compare with the rest of the processes

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsample

In [19]:
# Question 5: What are mean squared errors of RF model on the training set, 
# in cross-validation (cross_val_score with scoring=’neg_mean_squared_error’ and 
# other arguments left with default values) and on holdout set?

print("Mean squared error train", mean_squared_error(y_train, forest.predict(X_train_scaled)))
print("Mean squared error cv", np.mean(
    np.abs(
        cross_val_score(
            forest, X_train_scaled, y_train, scoring="neg_mean_squared_error"
        )
    )
))
print("Mean squared error holdout", mean_squared_error(y_holdout, forest.predict(X_holdout_scaled)))


Mean squared error train 0.05261155192532089
Mean squared error cv 0.4142003732204039
Mean squared error holdout 0.41078


Tune the max_features and max_depth hyperparameters with GridSearchCV and again check mean cross-validation MSE and MSE on holdout set.

In [20]:
forest_params = {'max_depth': list(range(10, 25)),
#                 'min_samples_leaf': list(range(1, 8)),
                 'max_features': list(range(6,12))}

locally_best_forest = GridSearchCV(RandomForestRegressor(n_jobs=-1, random_state=17),
                                    param_grid=forest_params,
                                     scoring="neg_mean_squared_error" ,
                                     cv=5,
                                     n_jobs=-1,
                                     verbose=True)
locally_best_forest.fit(X_train_scaled, y_train)

Fitting 5 folds for each of 90 candidates, totalling 450 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",RandomForestR...ndom_state=17)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'max_depth': [10, 11, ...], 'max_features': [6, 7, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_mean_squared_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2 : the score is al

In [21]:
locally_best_forest.best_score_, locally_best_forest.best_params_

(np.float64(-0.39773288191505934), {'max_depth': 21, 'max_features': 6})

## Why best score is negative

This is one of the most confusing things about scikit-learn at first, but it's intentional.

You used:

```python
scoring="neg_mean_squared_error"
```

Notice the **`neg_`** prefix.

### Why is it negative?

The **Mean Squared Error (MSE)** is defined as

[
\text{MSE} = \frac{1}{n}\sum (y_i - \hat{y}_i)^2
]

Since errors are squared:

* MSE is always **0 or positive**
* **Smaller is better**

For example:

| Model |  MSE |
| ----- | ---: |
| A     | 0.40 |
| B     | 0.25 |
| C     | 0.80 |

Model B is the best because it has the **lowest** MSE.

---

### But `GridSearchCV` always tries to maximize the score

Internally, `GridSearchCV` assumes:

> **Higher score = better model**

That works naturally for metrics like:

* Accuracy
* R²
* F1 score

But it doesn't work for MSE because **lower** is better.

So scikit-learn simply multiplies the MSE by **−1**.

|  MSE | Neg MSE |
| ---: | ------: |
| 0.25 |   -0.25 |
| 0.40 |   -0.40 |
| 0.80 |   -0.80 |

Now the "best" model is the one with the **largest** value:

```text
-0.25 > -0.40 > -0.80
```

which corresponds to the **smallest** MSE.

---

### In your case

You got:

```python
best_score_ = -0.3977328819150593
```

That means the actual cross-validated MSE is:

```python
MSE = 0.3977328819150593
```

You can simply negate it:

```python
mse = -locally_best_forest.best_score_
print(mse)
```

---

### Rule to remember

Any scoring metric in scikit-learn that starts with `neg_` means:

> **The real error is the negative of the reported score.**

For example:

* `neg_mean_squared_error`
* `neg_root_mean_squared_error`
* `neg_mean_absolute_error`

These are all reported as negative values so that `GridSearchCV` can consistently **maximize** the score instead of having to minimize some metrics and maximize others.


In [22]:
# Question 6: What are mean squared errors of tuned RF model in cross-validation 
# (cross_val_score with scoring=’neg_mean_squared_error’ and other arguments left with default values) 
# and on holdout set?

print("Mean squared error cv", np.mean(
    np.abs(
        cross_val_score(
            locally_best_forest.best_estimator_, X_train_scaled, y_train,scoring="neg_mean_squared_error"
        )
    )
))

print("Mean squared error test", mean_squared_error(y_holdout, locally_best_forest.predict(X_holdout_scaled)))

Mean squared error cv 0.3977328819150594
Mean squared error test 0.40224059108626947


Output RF’s feature importance. Again, it’s nice to present it as a DataFrame.

In [29]:
# Question 7: What is the most important feature, according to the Random Forest model?
rf_importance = pd.DataFrame(
    locally_best_forest.best_estimator_.feature_importances_,
    columns=["coeff"],
    index=data.columns[:-1]
)
rf_importance.sort_values(by="coeff", ascending=False)

,coeff
alcohol,0.206056
volatile acidity,0.117578
free sulfur dioxide,0.111556
density,0.088549
pH,0.073659
total sulfur dioxide,0.073640
chlorides,0.073366
residual sugar,0.072072
citric acid,0.062601
fixed acidity,0.061813
